# Chapter 12 Simulations — Feedback Stage: Grid Search, Ground-Truth Evaluation & Judge Calibration

Stage 7 closes the intelligence cycle. After dissemination (Stage 6), stakeholders return raw questions, concerns, and validation requests. The Stage 7 pipeline transforms that unstructured stakeholder feedback into atomic, structured follow-on QIRs that re-enter the cycle at Stage 1 (Requirement). A `Compliance_Judge` validates every QIR against the `QIR_TEMPLATE` (9 required fields), the four scope types (EXPANSION / DEEPENING / PIVOT / VALIDATION), the three priority levels, and the six transformation rules (TR-001 through TR-006). A `Verification_Agent` mechanically applies the judge's corrections, and a self-refining loop runs until the verdict is `PASS` or `max_iterations` is hit.

This notebook validates the pipeline at three layers — judge calibration, ground-truth evaluation, and configuration grid search — because no single layer catches every failure mode. A lenient judge can emit `PASS` on QIRs that miss half the stakeholder questions; an unmatched ground-truth scenario can hide a deterministic schema bug; a high-F1 grid-search winner can still misclassify scope on every refinement pass. Only running all three together produces a configuration that is safe to ship.

Roadmap:

- **Section 1** — Setup: load the pipeline, judge cases, and template constants.
- **Section 2** — Pipeline helper: a thin async wrapper around `run_pipeline()`.
- **Section 3** — Single-scenario demo: the canonical CISO three-question compound case.
- **Section 4** — Configuration grid search: model × temperature × scenario × Monte Carlo runs, ranked by judge F1.
- **Section 5** — Ground-truth evaluation: bipartite QIR matching against expert-curated `expected_qirs`.
- **Section 6** — Judge calibration: ten synthetic QIR arrays with known-correct verdicts.
- **Section 7** — Discussion: how the three layers compose into trustworthy production validation.


## 1. Setup

In [1]:
# Setup: load env, wire sys.path so the my_agent package imports cleanly,
# and bring in the pipeline + judge eval + template constants.
import asyncio
import json
import os
import sys
import csv
import re
import time
from pathlib import Path
from collections import defaultdict
from typing import Optional

import pandas as pd
from IPython.display import HTML, display

from dotenv import load_dotenv
load_dotenv()

sys.path.insert(0, str(Path("my_agent").resolve()))

from agent import (
    run_pipeline,
    DEFAULT_MODEL,
    DEFAULT_TEMP,
    APP_NAME,
    USER_ID,
    SAMPLE_STAKEHOLDER_FEEDBACK,
    SAMPLE_ORIGINAL_QIR,
)
from judge_eval import (
    TEST_CASES,
    run_judge_on_feedback_qirs,
    _score_case,
)
# JUDGE_ACCURACY_THRESHOLD lives inside judge_eval.main(); mirror it here.
try:
    from judge_eval import JUDGE_ACCURACY_THRESHOLD  # type: ignore
except ImportError:
    JUDGE_ACCURACY_THRESHOLD = 80.0

from requirement_templates import (
    QIR_TEMPLATE,
    SCOPE_TYPES,
    PRIORITY_LEVELS,
    TRANSFORMATION_RULES,
    compute_verdict_metrics,
)

print(f"Default model       : {DEFAULT_MODEL} @ temp={DEFAULT_TEMP}")
print(f"Judge cases         : {len(TEST_CASES)}")
print(f"QIR template fields : {len(QIR_TEMPLATE)}")
print(f"Scope types         : {len(SCOPE_TYPES)}")
print(f"Priority levels     : {len(PRIORITY_LEVELS)}")
print(f"Transformation rules: {len(TRANSFORMATION_RULES)}")
print(f"Judge threshold     : {JUDGE_ACCURACY_THRESHOLD:.0f}%")


Default model       : gemini-2.5-flash @ temp=0.2
Judge cases         : 10
QIR template fields : 9
Scope types         : 4
Priority levels     : 3
Transformation rules: 6
Judge threshold     : 80%


## 2. Pipeline Helper

Wraps a single end-to-end run of `run_pipeline()` from `agent.py`. The agent constructs fresh instances per call, so reusing the helper across the notebook is safe — each invocation gets its own session and `Runner`. The helper returns a flat dict of summary stats (verdict, QIR count, iteration count, latency, and the parsed QIR list) suitable for both the demo cell and the grid-search aggregator.


In [2]:
# Async wrapper around run_pipeline() that returns a flat result dict.
# Used by the demo cell and reused conceptually by the grid search runner.

async def run_scenario(
    stakeholder_feedback: str,
    original_qir: str = SAMPLE_ORIGINAL_QIR,
    max_iterations: int = 3,
    generator_model: str = DEFAULT_MODEL,
    generator_temp: float = DEFAULT_TEMP,
    verbose: bool = True,
) -> dict:
    """Run the feedback pipeline once and return summary stats."""
    if verbose:
        print(f"Model: {generator_model} @ temp={generator_temp}  max_iter={max_iterations}")
        print()

    start = time.monotonic()
    session_id, iterations = await run_pipeline(
        stakeholder_feedback=stakeholder_feedback,
        original_qir=original_qir,
        max_iterations=max_iterations,
        generator_model=generator_model,
        generator_temp=generator_temp,
    )
    latency_s = round(time.monotonic() - start, 2)

    final = iterations[-1]
    verdict = final["verdict"]
    verdict_label = verdict.get("verdict", "UNKNOWN")
    confirmed = verdict.get("confirmed_valid", [])
    unverified = verdict.get("unverified", [])
    missing = verdict.get("missing_critical", [])

    # Parse final QIRs from verified_feedback (or feedback_qirs fallback).
    qirs_raw = final.get("verified_feedback") or final.get("feedback_qirs") or "[]"
    try:
        qirs = json.loads(qirs_raw) if isinstance(qirs_raw, str) else qirs_raw
        if not isinstance(qirs, list):
            qirs = []
    except (json.JSONDecodeError, TypeError):
        qirs = []

    if verbose:
        for it in iterations:
            n = it["iteration"]
            v = it["verdict"]
            n_v = len(v.get("confirmed_valid", []))
            n_u = len(v.get("unverified", []))
            n_m = len(v.get("missing_critical", []))
            label = v.get("verdict", "UNKNOWN")
            print(f"{chr(0x2500)*60}")
            print(f"Iteration {n} \u2014 {label}")
            print(f"  valid={n_v}  unverified={n_u}  missing={n_m}")
            print(f"  {v.get('summary', '')[:120]}")
            print()
        print(f"{'='*60}")
        print(f"Final verdict : {verdict_label} in {len(iterations)} iteration(s)")
        print(f"QIRs produced : {len(qirs)}")
        print(f"Latency       : {latency_s}s")

    return {
        "session_id":   session_id,
        "verdict_label": verdict_label,
        "n_qirs":       len(qirs),
        "n_iters":      len(iterations),
        "latency_s":    latency_s,
        "confirmed":    len(confirmed),
        "unverified":   len(unverified),
        "missing":      len(missing),
        "qirs":         qirs,
    }


## 3. Single-Scenario Demo

Runs the canonical CISO compound feedback case: three distinct questions in one stakeholder message, drawn from Scenario 5 in `ground_truth.csv`. Each question must decompose into its own atomic QIR with the correct scope (EXPANSION, VALIDATION, DEEPENING) and priority. This demonstrates the full self-refining loop end-to-end: Transformation Agent generates QIRs, Compliance Judge validates, Verification Agent corrects, and the loop exits on `PASS` via the escalation callback.


In [ ]:
# Canonical demo: CISO compound feedback (3 questions) from ground_truth.csv Scenario 5.
DEMO_FEEDBACK = (
    "CISO compound feedback after Stage 6 briefing:\n\n"
    "1. Could 50 other vendors be compromised the same way? We use the same "
    "Okta SSO integration with all partner shops.\n\n"
    "2. Can we prevent session hijacking entirely with FIDO2? I want to know "
    "if it's worth the investment.\n\n"
    "3. Can we detect if the attacker planted backdoors in our repos? We need "
    "to know if the 12 repositories are clean."
)

demo_result = await run_scenario(
    stakeholder_feedback=DEMO_FEEDBACK,
    original_qir=SAMPLE_ORIGINAL_QIR,
    verbose=True,
)

print()
print(f"Summary: verdict={demo_result['verdict_label']}  "
      f"qirs={demo_result['n_qirs']}  iters={demo_result['n_iters']}  "
      f"latency={demo_result['latency_s']}s")


## 4. Configuration Grid Search

The sweep covers temperature × scenario × Monte Carlo runs at a fixed model. Each scenario exercises a distinct feedback shape — single question, compound, praise-only, multi-domain pivot — so the winning configuration generalises beyond the canonical CISO case.

**Monte Carlo suppression at temperature 0.0.** Repeated runs at `temp=0.0` are deterministic up to API-side nondeterminism, which is negligible compared to the cost of redundant API calls. The grid runner therefore executes only one run per scenario when `temp == 0.0`, and the full `MONTE_CARLO_RUNS` count when `temp > 0.0`. This is enforced by `runs_for_temp = MONTE_CARLO_RUNS if temp > 0 else 1` in the runner. Configurations are ranked by average F1 (judge-confirmed QIRs as a fraction of total QIRs); ties are broken by average iteration count. The winning configuration is written to `my_agent/best_config.json`.


In [ ]:
# Grid search constants and scenario definitions.
# Single model (chapter does not sweep providers); three temperatures; six scenarios.
import asyncio
import json
import csv as _csv_gs
import time
from pathlib import Path
from collections import defaultdict

GS_RESULTS_CSV = Path("config_search_results.csv")
GS_CHECKPOINT  = Path("gt_checkpoint.json")
CONCURRENCY_LIMIT = 5
MONTE_CARLO_RUNS  = 5

MODELS       = ["gemini-2.5-flash"]
TEMPERATURES = [0.0, 0.2, 0.5]

# Six abbreviated stakeholder feedback scenarios.
GS_SCENARIOS = [
    {
        "name": "Vendor Compromise Expansion",
        "stakeholder_feedback": (
            "CISO: Could 50 other vendors be compromised the same way? "
            "We use the same Okta SSO integration with all partner development shops."
        ),
    },
    {
        "name": "FIDO2 Validation",
        "stakeholder_feedback": (
            "CISO: Can we prevent session hijacking entirely with FIDO2? "
            "Would FIDO2/WebAuthn keys have stopped this and is it worth the investment?"
        ),
    },
    {
        "name": "Dwell Time + GDPR Compound",
        "stakeholder_feedback": (
            "SOC Manager: How long did the attacker have access before we detected it? "
            "Also, do we need to notify affected customers under GDPR?"
        ),
    },
    {
        "name": "Code Publication Impact",
        "stakeholder_feedback": (
            "VP Engineering: What's our exposure if the stolen source code from the "
            "12 repositories is published tomorrow? What's the worst case?"
        ),
    },
    {
        "name": "CISO Three-Question Compound",
        "stakeholder_feedback": (
            "CISO compound feedback:\n"
            "1. Could 50 other vendors be compromised the same way?\n"
            "2. Can we prevent session hijacking entirely with FIDO2?\n"
            "3. Can we detect if the attacker planted backdoors in our repos?"
        ),
    },
    {
        "name": "Praise Only (TR-006)",
        "stakeholder_feedback": (
            "VP Engineering: Great work on the threat briefing. The analysis was "
            "thorough and the recommendations were clear. The team appreciated it."
        ),
    },
]

total_runs = sum(
    len(GS_SCENARIOS) * (MONTE_CARLO_RUNS if temp > 0 else 1)
    for temp in TEMPERATURES
) * len(MODELS)

print(f"Models       : {len(MODELS)}  ({MODELS[0]})")
print(f"Temperatures : {TEMPERATURES}")
print(f"Scenarios    : {len(GS_SCENARIOS)}")
print(f"MC runs/temp : {MONTE_CARLO_RUNS} (1 at temp=0.0)")
print(f"Total runs   : {total_runs}")


In [ ]:
# Grid search runner with checkpoint resume.
# Skips work already present in config_search_runs.csv (the per-run checkpoint). Aggregates per
# (model, temperature) into avg_f1, avg_iters, pass_rate, avg_latency_s.

PER_RUN_CSV = Path("config_search_runs.csv")  # Per-run rows for resume.

PER_RUN_FIELDS = [
    "model", "temperature", "scenario_idx", "run_idx",
    "n_qirs", "confirmed", "unverified", "missing", "f1",
    "n_iters", "latency_s", "verdict",
]

def _load_completed():
    """Return set of (model, temp, scenario_idx, run_idx) already computed."""
    completed = set()
    rows = []
    if PER_RUN_CSV.exists() and PER_RUN_CSV.stat().st_size > 0 and not pd.read_csv(PER_RUN_CSV).empty:
        with PER_RUN_CSV.open(newline="", encoding="utf-8") as f:
            for r in _csv_gs.DictReader(f):
                key = (r["model"], float(r["temperature"]),
                       int(r["scenario_idx"]), int(r["run_idx"]))
                completed.add(key)
                rows.append(r)
    return completed, rows


async def _run_single(model, temp, scenario_idx, run_idx):
    """One pipeline execution. Returns a per-run dict."""
    scenario = GS_SCENARIOS[scenario_idx]
    start = time.monotonic()
    _, iterations = await run_pipeline(
        stakeholder_feedback=scenario["stakeholder_feedback"],
        original_qir=SAMPLE_ORIGINAL_QIR,
        max_iterations=3,
        generator_model=model,
        generator_temp=temp,
    )
    latency = round(time.monotonic() - start, 2)
    final = iterations[-1]["verdict"]
    confirmed  = len(final.get("confirmed_valid", []))
    unverified = len(final.get("unverified", []))
    missing    = len(final.get("missing_critical", []))
    n_qirs     = confirmed + unverified
    # F1 = harmonic mean of precision and coverage on the confirmed QIR set,
    # via the shared compute_verdict_metrics() helper (rescaled to 0-1).
    f1 = compute_verdict_metrics(final)["f1"] / 100.0
    return {
        "model": model, "temperature": temp,
        "scenario_idx": scenario_idx, "run_idx": run_idx,
        "n_qirs": n_qirs, "confirmed": confirmed,
        "unverified": unverified, "missing": missing,
        "f1": round(f1, 3), "n_iters": len(iterations),
        "latency_s": latency, "verdict": final.get("verdict", "UNKNOWN"),
    }


async def run_grid():
    completed, existing_rows = _load_completed()
    semaphore = asyncio.Semaphore(CONCURRENCY_LIMIT)

    async def bounded(model, temp, sidx, ridx):
        async with semaphore:
            return await _run_single(model, temp, sidx, ridx)

    tasks = []
    for model in MODELS:
        for temp in TEMPERATURES:
            runs_for_temp = MONTE_CARLO_RUNS if temp > 0 else 1
            for sidx in range(len(GS_SCENARIOS)):
                for ridx in range(1, runs_for_temp + 1):
                    if (model, temp, sidx, ridx) in completed:
                        continue
                    tasks.append(bounded(model, temp, sidx, ridx))

    print(f"Resuming from {len(completed)} completed run(s).")
    print(f"Scheduling {len(tasks)} new run(s) (concurrency={CONCURRENCY_LIMIT})...")

    new_rows = []
    done = 0
    if tasks:
        for coro in asyncio.as_completed(tasks):
            r = await coro
            new_rows.append(r)
            done += 1
            if done % 20 == 0 or done == len(tasks):
                print(f"  {done}/{len(tasks)} new run(s) complete")

    # Persist per-run CSV (existing + new) for future resume.
    all_rows = list(existing_rows) + new_rows
    with PER_RUN_CSV.open("w", newline="", encoding="utf-8") as f:
        w = _csv_gs.DictWriter(f, fieldnames=PER_RUN_FIELDS, quoting=_csv_gs.QUOTE_ALL)
        w.writeheader()
        for r in all_rows:
            w.writerow({k: r.get(k, "") for k in PER_RUN_FIELDS})

    # Aggregate by (model, temperature).
    agg = defaultdict(list)
    for r in all_rows:
        key = (r["model"], float(r["temperature"]))
        # Coerce numerics from CSV-read strings.
        agg[key].append({
            "f1":        float(r["f1"]),
            "n_iters":   int(r["n_iters"]),
            "latency_s": float(r["latency_s"]),
            "verdict":   r["verdict"],
        })

    summary_rows = []
    for (model, temp), runs in sorted(agg.items()):
        if not runs:
            continue
        avg_f1    = sum(x["f1"] for x in runs) / len(runs)
        avg_iters = sum(x["n_iters"] for x in runs) / len(runs)
        avg_lat   = sum(x["latency_s"] for x in runs) / len(runs)
        pass_rate = sum(1 for x in runs if x["verdict"] == "PASS") / len(runs)
        summary_rows.append({
            "model": model,
            "temperature": temp,
            "avg_f1":        round(avg_f1, 3),
            "avg_iters":     round(avg_iters, 2),
            "pass_rate":     round(pass_rate, 3),
            "avg_latency_s": round(avg_lat, 2),
            "n_runs":        len(runs),
        })

    summary_rows.sort(key=lambda x: (-x["avg_f1"], x["avg_iters"]))

    fieldnames = ["model", "temperature", "avg_f1", "avg_iters",
                  "pass_rate", "avg_latency_s", "n_runs"]
    with GS_RESULTS_CSV.open("w", newline="", encoding="utf-8") as f:
        w = _csv_gs.DictWriter(f, fieldnames=fieldnames, quoting=_csv_gs.QUOTE_ALL)
        w.writeheader()
        w.writerows(summary_rows)

    if summary_rows:
        best = summary_rows[0]
        print(f"\nBest config: {best['model']} @ temp={best['temperature']}  "
              f"avg_f1={best['avg_f1']:.3f}  avg_iters={best['avg_iters']:.2f}  "
              f"pass_rate={best['pass_rate']:.0%}")
    print(f"Wrote {GS_RESULTS_CSV} ({len(summary_rows)} configs)")
    return summary_rows


gs_summary = await run_grid()


In [ ]:
# Reload config_search_results.csv and render a ranked Styler table.

if not GS_RESULTS_CSV.exists() or GS_RESULTS_CSV.stat().st_size == 0 or pd.read_csv(GS_RESULTS_CSV).empty:
    print("No grid search results yet \u2014 run gs-run first.")
else:
    _df_gs = pd.read_csv(GS_RESULTS_CSV)
    _df_gs_disp = _df_gs.rename(columns={
        "model":         "Model",
        "temperature":   "Temp",
        "avg_f1":        "avg F1",
        "avg_iters":     "Iters",
        "pass_rate":     "Pass %",
        "avg_latency_s": "Latency (s)",
        "n_runs":        "Runs",
    }).reset_index(drop=True)
    _df_gs_disp.index = _df_gs_disp.index + 1  # 1-based rank.

    def _color_f1(v):
        if v >= 0.90: return "background-color: #dcfce7; color: #166534; font-weight: 600"
        if v >= 0.70: return "background-color: #fef9c3; color: #92400e"
        return "background-color: #fee2e2; color: #991b1b"

    def _hl_best(row):
        return ["background-color: #eff6ff"] * len(row) if row.name == 1 else [""] * len(row)

    _gs_tbl_styles = [
        {"selector": "caption", "props": [("font-size", "0.9rem"), ("font-weight", "600"),
                                            ("padding", "0.4rem 0"), ("text-align", "left")]},
        {"selector": "th",      "props": [("font-size", "0.78rem"), ("text-transform", "uppercase"),
                                            ("color", "#6b7280"), ("padding", "0.35rem 0.55rem")]},
        {"selector": "td",      "props": [("padding", "0.35rem 0.55rem"), ("font-size", "0.83rem")]},
    ]

    display(
        _df_gs_disp.style
        .apply(_hl_best, axis=1)
        .map(_color_f1, subset=["avg F1"])
        .format({"avg F1": "{:.3f}", "Pass %": "{:.0%}",
                 "Iters": "{:.2f}", "Latency (s)": "{:.1f}s",
                 "Temp": "{:.1f}"})
        .set_caption(f"Configuration Grid Search \u2014 {len(_df_gs)} configurations, ranked by avg F1")
        .set_table_styles(_gs_tbl_styles)
    )


In [ ]:
# Model x Temperature F1 pivot table.

if not GS_RESULTS_CSV.exists() or GS_RESULTS_CSV.stat().st_size == 0 or pd.read_csv(GS_RESULTS_CSV).empty:
    print("No grid search results yet.")
else:
    _df_pivot = _df_gs.pivot_table(
        index="model", columns="temperature", values="avg_f1", aggfunc="mean"
    ).round(3)
    _df_pivot.index.name = "Model"
    _df_pivot.columns.name = "Temperature"

    def _color_cell(v):
        if pd.isna(v): return ""
        if v >= 0.90: return "background-color: #dcfce7; color: #166534; font-weight: 600"
        if v >= 0.70: return "background-color: #fef9c3; color: #92400e"
        return "background-color: #fee2e2; color: #991b1b"

    display(
        _df_pivot.style
        .map(_color_cell)
        .format("{:.3f}")
        .set_caption("avg F1 by Model \u00d7 Temperature")
    )


In [ ]:
# Persist the top-ranked config to my_agent/best_config.json (cwd-relative).

_best_path = Path("my_agent/best_config.json")
if not GS_RESULTS_CSV.exists() or GS_RESULTS_CSV.stat().st_size == 0 or pd.read_csv(GS_RESULTS_CSV).empty:
    print("No grid search results yet \u2014 cannot write best_config.json.")
else:
    _df_best = pd.read_csv(GS_RESULTS_CSV)
    if _df_best.empty:
        print("config_search_results.csv has no rows.")
    else:
        _top = _df_best.iloc[0]
        best = {
            "model":            str(_top["model"]),
            "temperature":      float(_top["temperature"]),
            "template_version": "1.0.0",
        }
        _best_path.write_text(json.dumps(best, indent=2))
        print(f"Wrote {_best_path}")
        print(json.dumps(best, indent=2))


## 5. Ground-Truth Evaluation

Each scenario in `ground_truth.csv` ships with an `expected_qirs` JSON array. Every expected QIR has a `label`, `scope`, `priority`, `description`, `key_phrases` (substrings that must appear somewhere in the generated `refined_requirement`), and an optional `temporal_window_contains` substring. The evaluator runs the pipeline, then performs **bipartite matching** between generated QIRs and expected QIRs by descending similarity score, with scope as a hard gate.

The scoring breakdown — recall, scope accuracy, priority accuracy, field completeness, and per-scope accuracy — surfaces *which axis* the pipeline drifts on. A high recall but low scope accuracy means the pipeline is capturing the right questions but misclassifying their scope; a high scope accuracy but low recall means clean QIRs are being generated for only some of the questions.


In [3]:
# Load ground-truth scenarios from ground_truth.csv.
GT_CSV = Path("ground_truth.csv")

GROUND_TRUTH = []
with GT_CSV.open(newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        try:
            expected = json.loads(row["expected_qirs"])
        except (json.JSONDecodeError, KeyError):
            expected = []
        GROUND_TRUTH.append({
            "name":         row["Scenario"],
            "stakeholder_feedback": row["stakeholder_feedback"],
            "expected_qirs": expected,
            "expected_qir_count": int(row.get("expected_qir_count", 0) or 0),
        })

print(f"Loaded {len(GROUND_TRUTH)} ground-truth scenarios from {GT_CSV}")
for i, gt in enumerate(GROUND_TRUTH, 1):
    print(f"  {i}. {gt['name'][:60]}  (expected {gt['expected_qir_count']} QIR(s))")

# Deployment gates (mirrored in gt-run; defined here so the CSV re-render
# path works on a fresh kernel without an API run).
RECALL_THRESHOLD = 0.60
SCOPE_THRESHOLD  = 70.0


Loaded 5 ground-truth scenarios from ground_truth.csv
  1. Vendor Compromise Expansion  (expected 1 QIR(s))
  2. FIDO2 Validation  (expected 1 QIR(s))
  3. Compound: Dwell Time + GDPR Notification  (expected 2 QIR(s))
  4. Code Publication Impact  (expected 1 QIR(s))
  5. CISO Compound: Three Questions  (expected 3 QIR(s))


In [4]:
# Bipartite QIR matching with scope hard-gate and phrase-fraction scoring.

def _score_qir_against_expected(actual: dict, expected: dict) -> float:
    """Return a [0..1] score for actual vs expected QIR. Scope mismatch returns 0."""
    actual_scope    = (actual.get("scope") or "").upper().strip()
    expected_scope  = (expected.get("scope") or "").upper().strip()
    if actual_scope != expected_scope:
        return 0.0  # Hard gate.

    score = 0.4  # Base for scope match.

    # Priority match: 0.2.
    if (actual.get("priority") or "").upper().strip() == (expected.get("priority") or "").upper().strip():
        score += 0.2

    # Phrase coverage: up to 0.4 for fraction of key_phrases found in refined_requirement.
    phrases = expected.get("key_phrases", []) or []
    if phrases:
        haystack = ((actual.get("refined_requirement") or "")
                    + " " + (actual.get("raw_feedback") or "")).lower()
        hits = sum(1 for p in phrases if p.lower() in haystack)
        score += 0.4 * (hits / len(phrases))
    else:
        score += 0.4  # No phrases to check → give credit.

    return round(score, 4)


def compare_to_ground_truth(generated_qirs: list, gt: dict) -> dict:
    """
    Bipartite match generated QIRs to expected QIRs (greedy, by descending score),
    then compute recall/precision/F1 and per-axis accuracy on matched pairs.
    """
    expected = gt.get("expected_qirs", []) or []
    generated = generated_qirs or []

    # Score every (gen, exp) pair.
    pair_scores = []
    for gi, gen in enumerate(generated):
        for ei, exp in enumerate(expected):
            s = _score_qir_against_expected(gen, exp)
            if s >= 0.3:  # Acceptance threshold.
                pair_scores.append((s, gi, ei))
    pair_scores.sort(key=lambda t: -t[0])

    used_gen, used_exp = set(), set()
    matches = []  # list of (gi, ei, score)
    for s, gi, ei in pair_scores:
        if gi in used_gen or ei in used_exp:
            continue
        used_gen.add(gi); used_exp.add(ei)
        matches.append((gi, ei, s))

    n_matched   = len(matches)
    n_expected  = len(expected)
    n_generated = len(generated)

    recall    = n_matched / n_expected  if n_expected  else 1.0
    precision = n_matched / n_generated if n_generated else 0.0
    f1        = (2 * recall * precision / (recall + precision)) if (recall + precision) else 0.0

    # Per-axis accuracy on matched pairs.
    scope_hits = priority_hits = field_hits = 0
    REQUIRED = list(QIR_TEMPLATE.keys())
    per_scope_total = defaultdict(int)
    per_scope_hits  = defaultdict(int)

    for gi, ei, _ in matches:
        gen = generated[gi]
        exp = expected[ei]
        exp_scope = (exp.get("scope") or "").upper()
        per_scope_total[exp_scope] += 1
        if (gen.get("scope") or "").upper() == exp_scope:
            scope_hits += 1
            per_scope_hits[exp_scope] += 1
        if (gen.get("priority") or "").upper() == (exp.get("priority") or "").upper():
            priority_hits += 1
        # Field completeness: all 9 required fields non-empty.
        if all((gen.get(k) or "") != "" for k in REQUIRED):
            field_hits += 1

    scope_accuracy      = (scope_hits     / n_matched * 100.0) if n_matched else 0.0
    priority_accuracy   = (priority_hits  / n_matched * 100.0) if n_matched else 0.0
    field_completeness  = (field_hits     / n_matched * 100.0) if n_matched else 0.0

    per_scope_accuracy = {
        s: (per_scope_hits[s] / per_scope_total[s] * 100.0) if per_scope_total[s] else 0.0
        for s in per_scope_total
    }

    matched_exp_idxs = {ei for _, ei, _ in matches}
    matched_gen_idxs = {gi for gi, _, _ in matches}
    missed_expected = [expected[i].get("label", f"exp_{i}") for i in range(n_expected) if i not in matched_exp_idxs]
    extra_generated = [generated[i].get("qir_id", f"gen_{i}") for i in range(n_generated) if i not in matched_gen_idxs]

    return {
        "recall":             round(recall, 3),
        "precision":          round(precision, 3),
        "f1":                 round(f1, 3),
        "scope_accuracy":     round(scope_accuracy, 1),
        "priority_accuracy":  round(priority_accuracy, 1),
        "field_completeness": round(field_completeness, 1),
        "n_matched":          n_matched,
        "n_expected":         n_expected,
        "n_generated":        n_generated,
        "missed_expected":    missed_expected,
        "extra_generated":    extra_generated,
        "per_scope_accuracy": {k: round(v, 1) for k, v in per_scope_accuracy.items()},
    }

print("Ground-truth helpers loaded.")


Ground-truth helpers loaded.


In [5]:
# Run the pipeline on each ground-truth scenario and score against expected QIRs.

RECALL_THRESHOLD = 0.60
SCOPE_THRESHOLD  = 70.0

gt_results = []

for i, gt in enumerate(GROUND_TRUTH, 1):
    print(f"\n{chr(0x2500)*60}")
    print(f"Scenario {i}/{len(GROUND_TRUTH)}: {gt['name']}")

    _, iterations = await run_pipeline(
        stakeholder_feedback=gt["stakeholder_feedback"],
        original_qir=SAMPLE_ORIGINAL_QIR,
        max_iterations=3,
    )

    final = iterations[-1]
    verdict_label = final["verdict"].get("verdict", "UNKNOWN")
    n_iters = len(iterations)

    qirs_raw = final.get("verified_feedback") or final.get("feedback_qirs") or "[]"
    try:
        generated_qirs = json.loads(qirs_raw) if isinstance(qirs_raw, str) else qirs_raw
        if not isinstance(generated_qirs, list):
            generated_qirs = []
    except (json.JSONDecodeError, TypeError):
        generated_qirs = []

    cmp = compare_to_ground_truth(generated_qirs, gt)
    gt_results.append({
        "name":     gt["name"],
        "n_iters":  n_iters,
        "verdict":  verdict_label,
        "cmp":      cmp,
    })

    print(f"  Verdict       : {verdict_label} in {n_iters} iter(s)")
    print(f"  Expected      : {cmp['n_expected']} QIR(s)   "
          f"Generated: {cmp['n_generated']}   Matched: {cmp['n_matched']}")
    print(f"  Recall        : {cmp['recall']:.1%}   Precision: {cmp['precision']:.1%}   "
          f"F1: {cmp['f1']:.3f}")
    print(f"  Scope acc     : {cmp['scope_accuracy']:.0f}%   "
          f"Priority acc: {cmp['priority_accuracy']:.0f}%   "
          f"Fields: {cmp['field_completeness']:.0f}%")
    if cmp["missed_expected"]:
        print(f"  Missed ({len(cmp['missed_expected'])}): {cmp['missed_expected']}")
    if cmp["extra_generated"]:
        print(f"  Extra  ({len(cmp['extra_generated'])}): {cmp['extra_generated']}")

print(f"\n{'='*60}")
print(f"GROUND-TRUTH EVALUATION COMPLETE \u2014 {len(gt_results)} scenario(s)")
print(f"{'='*60}")



────────────────────────────────────────────────────────────
Scenario 1/5: Vendor Compromise Expansion
  Iteration 1/3: PASS — All QIRs are valid and all distinct questions from the feedback have been captured.
  Verdict       : PASS in 1 iter(s)
  Expected      : 1 QIR(s)   Generated: 1   Matched: 1
  Recall        : 100.0%   Precision: 100.0%   F1: 1.000
  Scope acc     : 100%   Priority acc: 100%   Fields: 100%

────────────────────────────────────────────────────────────
Scenario 2/5: FIDO2 Validation
  Iteration 1/3: FAIL — One QIR failed validation due to a vague temporal window, resulting in a FAIL verdict.
  Iteration 2/3: PASS — All QIRs are valid, complete, and accurately capture the distinct questions from the stakeholder feedback, resolving previous issues.
  Verdict       : PASS in 2 iter(s)
  Expected      : 1 QIR(s)   Generated: 3   Matched: 1
  Recall        : 100.0%   Precision: 33.3%   F1: 0.500
  Scope acc     : 100%   Priority acc: 100%   Fields: 100%
  Extra  (2):

In [6]:
# Aggregate gt_results, write per-scenario rows back into ground_truth.csv,
# and threshold-check.

if not globals().get("gt_results"):  # undefined on a fresh kernel until gt-run executes
    print("No gt_results yet \u2014 run gt-run first.")
else:
    avg_recall            = sum(r["cmp"]["recall"]            for r in gt_results) / len(gt_results)
    avg_scope_accuracy    = sum(r["cmp"]["scope_accuracy"]    for r in gt_results) / len(gt_results)
    avg_priority_accuracy = sum(r["cmp"]["priority_accuracy"] for r in gt_results) / len(gt_results)
    avg_field_completeness = sum(r["cmp"]["field_completeness"] for r in gt_results) / len(gt_results)
    avg_f1                = sum(r["cmp"]["f1"]                for r in gt_results) / len(gt_results)

    print(f"avg Recall              : {avg_recall:.1%}    (threshold >= {RECALL_THRESHOLD:.0%})")
    print(f"avg Scope accuracy      : {avg_scope_accuracy:.0f}%   (threshold >= {SCOPE_THRESHOLD:.0f}%)")
    print(f"avg Priority accuracy   : {avg_priority_accuracy:.0f}%")
    print(f"avg Field completeness  : {avg_field_completeness:.0f}%")
    print(f"avg F1                  : {avg_f1:.3f}")
    print()

    # Write back to ground_truth.csv by Scenario lookup.
    existing = []
    with GT_CSV.open(newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        fieldnames = reader.fieldnames
        for r in reader:
            existing.append(r)

    by_name = {r["name"]: r for r in gt_results}
    for row in existing:
        r = by_name.get(row["Scenario"])
        if r is None:
            continue
        c = r["cmp"]
        row["Pipeline_QIRs"]      = c["n_generated"]
        row["GT_QIRs"]            = c["n_expected"]
        row["Matched"]            = c["n_matched"]
        row["Recall"]             = c["recall"]
        row["Scope_Accuracy"]     = c["scope_accuracy"]
        row["Priority_Accuracy"]  = c["priority_accuracy"]
        row["Field_Completeness"] = c["field_completeness"]
        row["F1"]                 = c["f1"]
        row["Verdict"]            = r["verdict"]
        row["Iters"]              = r["n_iters"]

    with GT_CSV.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames, quoting=csv.QUOTE_ALL)
        w.writeheader()
        w.writerows(existing)
    print(f"Wrote per-scenario rows to {GT_CSV}")

    # Threshold check.
    failed = False
    if avg_recall < RECALL_THRESHOLD:
        print(f"Recall {avg_recall:.1%} BELOW THRESHOLD ({RECALL_THRESHOLD:.0%}) \u2014 FAIL"); failed = True
    if avg_scope_accuracy < SCOPE_THRESHOLD:
        print(f"Scope accuracy {avg_scope_accuracy:.0f}% BELOW THRESHOLD ({SCOPE_THRESHOLD:.0f}%) \u2014 FAIL"); failed = True
    if not failed:
        print(f"ALL THRESHOLDS MET \u2014 PASS")


avg Recall              : 80.0%    (threshold >= 60%)
avg Scope accuracy      : 80%   (threshold >= 70%)
avg Priority accuracy   : 80%
avg Field completeness  : 80%
avg F1                  : 0.700

Wrote per-scenario rows to ground_truth.csv
ALL THRESHOLDS MET — PASS


In [7]:
# Styled per-scenario ground-truth results table, reloaded from ground_truth.csv.

_gt_rows = []
with GT_CSV.open(newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        if (row.get("Verdict") or "").strip().upper() in ("", "PENDING"):
            # Skip unrun rows (template rows ship with Verdict=PENDING).
            # NOTE: do NOT skip on zero scores — a genuinely run scenario can
            # legitimately score 0.0 (see Code Publication Impact).
            continue
        _gt_rows.append({
            "Scenario":  row["Scenario"][:55],
            "Exp":       int(float(row.get("GT_QIRs") or 0)),
            "Gen":       int(float(row.get("Pipeline_QIRs") or 0)),
            "Matched":   int(float(row.get("Matched") or 0)),
            "Recall":    float(row.get("Recall") or 0.0),
            "Scope %":   float(row.get("Scope_Accuracy") or 0.0),
            "Priority %": float(row.get("Priority_Accuracy") or 0.0),
            "Fields %":  float(row.get("Field_Completeness") or 0.0),
            "F1":        float(row.get("F1") or 0.0),
            "Verdict":   row.get("Verdict") or "",
            "Iters":     int(float(row.get("Iters") or 0)),
        })

if not _gt_rows:
    print("No ground-truth results yet \u2014 run gt-run + gt-summary first.")
else:
    _df_gt = pd.DataFrame(_gt_rows)

    def _color_pct(v):
        if v >= 0.90: return "background-color: #dcfce7; color: #166534; font-weight: 600"
        if v >= 0.70: return "background-color: #fef9c3; color: #92400e"
        return "background-color: #fee2e2; color: #991b1b"

    def _color_pct100(v):
        return _color_pct(v / 100.0)

    def _verdict_color(v):
        return {
            "PASS":    "color: #15803d; font-weight: 600",
            "PARTIAL": "color: #ca8a04; font-weight: 600",
            "FAIL":    "color: #dc2626; font-weight: 600",
        }.get(str(v), "color: #64748b")

    _gt_tbl_styles = [
        {"selector": "caption", "props": [("font-size", "0.9rem"), ("font-weight", "600"),
                                            ("padding", "0.4rem 0"), ("text-align", "left")]},
        {"selector": "th",      "props": [("font-size", "0.78rem"), ("text-transform", "uppercase"),
                                            ("color", "#6b7280"), ("padding", "0.35rem 0.55rem")]},
        {"selector": "td",      "props": [("padding", "0.35rem 0.55rem"), ("font-size", "0.83rem")]},
    ]

    display(
        _df_gt.style
        .map(_color_pct, subset=["Recall", "F1"])
        .map(_color_pct100, subset=["Scope %", "Priority %", "Fields %"])
        .map(_verdict_color, subset=["Verdict"])
        .format({"Recall": "{:.0%}", "F1": "{:.3f}",
                 "Scope %": "{:.0f}%", "Priority %": "{:.0f}%", "Fields %": "{:.0f}%"})
        .set_caption(
            f"Ground-Truth Evaluation \u2014 {len(_df_gt)} scenarios  |  "
            f"Recall \u2265 {RECALL_THRESHOLD:.0%}  |  Scope \u2265 {SCOPE_THRESHOLD:.0f}%"
        )
        .set_table_styles(_gt_tbl_styles)
    )


,Scenario,Exp,Gen,Matched,Recall,Scope %,Priority %,Fields %,F1,Verdict,Iters
0,Vendor Compromise Expansion,1,1,1,100%,100%,100%,100%,1.000,PASS,1
1,FIDO2 Validation,1,3,1,100%,100%,100%,100%,0.500,PASS,2
2,Compound: Dwell Time + GDPR Notification,2,2,2,100%,100%,100%,100%,1.000,PASS,1
3,Code Publication Impact,1,2,0,0%,0%,0%,0%,0.000,PASS,1
4,CISO Compound: Three Questions,3,3,3,100%,100%,100%,100%,1.000,PASS,1


## 6. Judge Calibration

Ten test cases with known-correct verdicts validate the Compliance Judge before its scores are trusted by the grid search. Each case fixes a synthetic QIR array and a stakeholder feedback string, then asserts the judge produces a specific verdict (`PASS` / `PARTIAL` / `FAIL`) and a minimum violation count. The cases cover every transformation rule (TR-001 through TR-006), template-completeness violations, and at least two PASS cases to catch over-strict judges.

**Threshold:** average accuracy across all cases must meet `JUDGE_ACCURACY_THRESHOLD` (80%).


In [8]:
# Run the Compliance Judge against each TEST_CASE and accumulate per-case scores.

print("=" * 70)
print("JUDGE CALIBRATION \u2014 Feedback Stage")
print(f"Test cases : {len(TEST_CASES)}")
print(f"Threshold  : {JUDGE_ACCURACY_THRESHOLD:.0f}%")
print("=" * 70)

eval_rows = []
total_pass = total_fail = total_checks = schema_failures = 0

for i, case in enumerate(TEST_CASES, 1):
    print(f"\n{chr(0x2500)*60}")
    print(f"Case {i}/{len(TEST_CASES)}: {case['name']}")

    result = await run_judge_on_feedback_qirs(
        feedback_qirs_text=case["feedback_qirs"],
        stakeholder_feedback=case["stakeholder_feedback"],
    )

    if not result["valid"]:
        print(f"  SCHEMA FAILURE: {result.get('error', '')[:200]}")
        schema_failures += 1
        total_fail   += 1
        total_checks += 1
        eval_rows.append({
            "name":             case["name"],
            "expected_verdict": case.get("expected_verdict") or "/".join(case.get("expected_verdict_options", [])),
            "actual_verdict":   "ERROR",
            "n_pass":           0,
            "n_fail":           1,
            "passed":           False,
            "summary":          (result.get("error") or "")[:80],
        })
        continue

    verdict = result["verdict"]
    scores  = _score_case(case, result)
    n_p = len(scores["passed_checks"])
    n_f = len(scores["failed_checks"])
    total_pass   += n_p
    total_fail   += n_f
    total_checks += n_p + n_f

    actual_verdict = verdict.get("verdict", "UNKNOWN")
    expected = case.get("expected_verdict") or "/".join(case.get("expected_verdict_options", []))

    print(f"  Expected : {expected}")
    print(f"  Actual   : {actual_verdict}   pass={n_p}  fail={n_f}")
    for chk in scores["passed_checks"]:
        print(f"    + {chk}")
    for chk in scores["failed_checks"]:
        print(f"    - {chk}")

    eval_rows.append({
        "name":             case["name"],
        "expected_verdict": expected,
        "actual_verdict":   actual_verdict,
        "n_pass":           n_p,
        "n_fail":           n_f,
        "passed":           n_f == 0,
        "summary":          verdict.get("summary", "")[:100],
    })

accuracy = (total_pass / total_checks * 100.0) if total_checks > 0 else 0.0
status   = "PASS" if accuracy >= JUDGE_ACCURACY_THRESHOLD else "FAIL"

print(f"\n{'='*70}")
print(f"Checks passed : {total_pass}/{total_checks}  "
      f"({total_fail} failed, {schema_failures} schema failure(s))")
print(f"Accuracy      : {accuracy:.0f}%   threshold {JUDGE_ACCURACY_THRESHOLD:.0f}%   \u2014 {status}")
print("=" * 70)


JUDGE CALIBRATION — Feedback Stage
Test cases : 10
Threshold  : 80%

────────────────────────────────────────────────────────────
Case 1/10: single_clear_question
  Expected : PASS
  Actual   : PASS   pass=2  fail=0
    + Verdict correct: PASS
    + Violation count correct: 0 (unverified=0, missing=0)

────────────────────────────────────────────────────────────
Case 2/10: compound_not_split
  Expected : FAIL
  Actual   : PARTIAL   pass=1  fail=1
    + Violation count adequate: 2 >= 1 (unverified=1, missing=1)
    - Verdict wrong: got PARTIAL, expected FAIL

────────────────────────────────────────────────────────────
Case 3/10: missing_temporal_window
  Expected : FAIL
  Actual   : FAIL   pass=2  fail=0
    + Verdict correct: FAIL
    + Violation count adequate: 1 >= 1 (unverified=1, missing=0)

────────────────────────────────────────────────────────────
Case 4/10: missing_linked_qir
  Expected : FAIL
  Actual   : FAIL   pass=2  fail=0
    + Verdict correct: FAIL
    + Violation coun

In [9]:
# Styled per-case judge calibration results table.

def _style_judge_eval(rows: list) -> "pd.io.formats.style.Styler":
    df = pd.DataFrame([{
        "Test Case": r["name"],
        "Expected":  r["expected_verdict"],
        "Actual":    r["actual_verdict"],
        "Pass":      r["n_pass"],
        "Fail":      r["n_fail"],
        "Result":    "OK" if r["passed"] else "FAIL",
        "Summary":   r["summary"],
    } for r in rows])

    n_pass  = sum(1 for r in rows if r["passed"])
    n_total = len(rows)
    acc     = (n_pass / n_total * 100.0) if n_total else 0.0
    cap_status = "PASS" if acc >= JUDGE_ACCURACY_THRESHOLD else "FAIL"

    def _row_color(row):
        return (["background-color: #dcfce7"] * len(row) if row["Result"] == "OK"
                else ["background-color: #fee2e2"] * len(row))

    def _verdict_color(val):
        return {
            "PASS":    "color: #15803d; font-weight: 600",
            "PARTIAL": "color: #ca8a04; font-weight: 600",
            "FAIL":    "color: #dc2626; font-weight: 600",
            "ERROR":   "color: #7c3aed; font-weight: 600",
        }.get(str(val), "color: #64748b")

    return (
        df.style
        .apply(_row_color, axis=1)
        .map(_verdict_color, subset=["Expected", "Actual"])
        .set_caption(
            f"Judge Calibration \u2014 {n_pass}/{n_total} cases fully correct "
            f"({acc:.0f}%) \u2014 threshold {JUDGE_ACCURACY_THRESHOLD:.0f}% \u2014 {cap_status}"
        )
        .set_table_styles([
            {"selector": "caption",
             "props": [("font-size", "0.9rem"), ("font-weight", "700"),
                       ("color", "#1e293b"), ("padding-bottom", "10px"),
                       ("text-align", "left")]},
            {"selector": "th",
             "props": [("background-color", "#f1f5f9"), ("color", "#475569"),
                       ("font-size", "0.82rem"), ("padding", "8px 12px"),
                       ("border-bottom", "2px solid #e2e8f0")]},
            {"selector": "td",
             "props": [("font-size", "0.83rem"), ("padding", "7px 12px"),
                       ("border-bottom", "1px solid #f1f5f9"),
                       ("max-width", "320px"), ("word-wrap", "break-word")]},
        ])
        .hide(axis="index")
    )

if not globals().get("eval_rows"):  # undefined on a fresh kernel until judge-run executes
    print("No judge eval rows yet \u2014 run judge-run first.")
else:
    display(_style_judge_eval(eval_rows))


Test Case,Expected,Actual,Pass,Fail,Result,Summary
single_clear_question,PASS,PASS,2,0,OK,"All QIRs are valid, complete, and accurately reflect the stakeholder feedback."
compound_not_split,FAIL,PARTIAL,1,1,FAIL,"One distinct question from the stakeholder feedback was not captured as a separate QIR, violating TR"
missing_temporal_window,FAIL,FAIL,2,0,OK,"One QIR failed validation due to an empty temporal window field, violating TR-005."
missing_linked_qir,FAIL,FAIL,2,0,OK,One QIR failed validation due to a missing required field.
two_questions_properly_split,PASS,PASS,2,0,OK,"All distinct questions from the stakeholder feedback were captured as separate QIRs, and all submitt"
unjustified_priority,FAIL,FAIL,2,0,OK,The QIR failed validation due to an empty priority justification.
wrong_scope_classification,PARTIAL/FAIL,PARTIAL,2,0,OK,One QIR has a scope misclassification (TR-003 violation).
praise_no_question,FAIL,FAIL,2,0,OK,One QIR failed validation due to generation from non-actionable feedback and a vague temporal window
five_questions_decomposed,PASS,PASS,2,0,OK,"All 5 follow-on QIRs are complete, correctly classified, justified, and explicitly scoped, capturing"
already_answered_question,PASS,PASS,3,0,OK,"All QIRs are valid, and all distinct questions from the stakeholder feedback have been captured."


## 7. Discussion

The three evaluations form a layered validation hierarchy, each catching failure modes the others cannot.

**Judge calibration** validates the Compliance Judge's own reliability before any of its scores are used downstream. The deterministic `check_verdict_rules` post-pass catches the most dangerous class of judge errors — `PASS` verdicts emitted while `unverified` or `missing_critical` are non-empty — but it cannot catch verdict logic that is internally consistent yet semantically wrong (a judge that accepts a praise-only QIR as `PASS` because every required field is present). Calibration cases like `praise_no_question` (TR-006) and `wrong_scope_classification` (TR-003) close that gap. If the judge fails calibration, every F1 score in the grid search is meaningless, so calibration runs first in production.

**Ground-truth evaluation** breaks the judge's monopoly on correctness. A lenient judge can emit `PASS` on a QIR set that omits half the stakeholder questions or misclassifies every scope. The bipartite-matching evaluator is the only layer that compares pipeline output to expert-curated `expected_qirs`: recall surfaces missing decompositions, scope accuracy surfaces classification drift, priority accuracy surfaces severity miscalibration, and field completeness surfaces template-schema regressions. The per-scope accuracy breakdown then localises where the drift is — a model that handles EXPANSION correctly but loses VALIDATION classification under refinement is a different fix than one that drops PIVOT scopes entirely.

**Configuration grid search** picks the production config from a sweep of temperatures and (in larger experiments) models. The chapter sweeps three temperatures — `0.0`, `0.2`, `0.5` — at a single model. Monte Carlo repeats are suppressed at `temp=0.0` because the model is deterministic up to negligible API-side jitter and re-running burns budget without producing new information. Configurations are ranked by average judge F1 with iteration count as the tiebreaker. The winning config is written to `my_agent/best_config.json` along with `template_version`, so a future schema bump (e.g. a new transformation rule) is detected by the stale-config guard in `agent.py` and forces a re-run.

Together, the three layers compose: calibration validates the judge, ground-truth validates the pipeline against human-curated truth, and grid search validates the production configuration against the calibrated judge on diverse inputs. None of the three is sufficient alone; running all three is the minimum bar for shipping a self-refining LLM pipeline that touches stakeholder feedback in a regulated workflow.
